In [ ]:
import os
import zarr
import numpy as np
import dask.array as da
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from matplotlib import colors, patches
import matplotlib.image as mpimg

In [ ]:
raw_data_dir = '/home/laura/Scriptie/ruwe_data'
bewerkte_data_dir = '/home/laura/Scriptie/bewerkte_data/BurntArea'

os.makedirs(bewerkte_data_dir, exist_ok=True)

path_phiSat2 = os.path.join(raw_data_dir, 'burned.zarr.zip')

MAX_PATCHES = 50

try:
    store = zarr.ZipStore(path_phiSat2, mode='r')
    zarr_root = zarr.group(store=store)
    trainval_group = zarr_root['trainval']
    
    all_X = []
    all_y = []
    
    patch_keys = list(trainval_group.keys())
    
    for i, key in enumerate(patch_keys[:MAX_PATCHES]):
        patch = trainval_group[key]
        
        # The [:] loads the data into RAM as array
        img_array = patch['img'][:]
        label_array = patch['label'][:]
        
        # Reshaping the data to coloms for RF
        img_reshaped = np.transpose(img_array, (1, 2, 0))
        
        # Reshaping label (from 4 to 1 channel)
        label_index = np.argmax(label_array, axis=0)
        
        # Flattening the data
        X_flat = img_reshaped.reshape(-1, 7) 
        y_flat = label_index.reshape(-1)
        
        all_X.append(X_flat)
        all_y.append(y_flat)
        

    X_dataset = np.vstack(all_X)
    y_dataset = np.concatenate(all_y)
    
    print(f"\n --- TOTAL PhiSat2 DATASET ---")
    print(f" - X (Features) matrix    : {X_dataset.shape} (Features)")
    print(f" - y (Labels) list       : {y_dataset.shape} (Labels)")

    np.save(os.path.join(bewerkte_data_dir, 'X_train_flat.npy'), X_dataset)
    np.save(os.path.join(bewerkte_data_dir, 'y_train_flat.npy'), y_dataset)

except Exception as e:
    print(f"Error: {e}")


 --- TOTAL PhiSat2 DATASET ---
 - X (Features) matrix    : (3276800, 7) (Features)
 - y (Labels) list       : (3276800,) (Labels)
